# Analiza Cererii de Decontare a Cheltuielilor

Acest notebook demonstrează cum să creați agenți care utilizează pluginuri pentru a procesa cheltuielile de călătorie din imagini locale ale chitanțelor, să genereze un email de cerere de decontare a cheltuielilor și să vizualizeze datele despre cheltuieli folosind un grafic circular. Agenții aleg dinamic funcțiile în funcție de contextul sarcinii.

Pași:
1. Agentul OCR procesează imaginea locală a chitanței și extrage datele despre cheltuielile de călătorie.
2. Agentul Email generează un email de cerere de decontare a cheltuielilor.

### Exemplu de scenariu pentru cheltuieli de călătorie:
Imaginați-vă că sunteți un angajat care călătorește pentru o întâlnire de afaceri într-un alt oraș. Compania dvs. are o politică de rambursare a tuturor cheltuielilor rezonabile legate de călătorie. Iată o defalcare a potențialelor cheltuieli de călătorie:
- Transport:
Bilet de avion dus-întors de la orașul dvs. de origine până la orașul destinație.
Taxi sau servicii de ride-hailing către și de la aeroport.
Transport local în orașul destinație (precum transport public, mașini închiriate sau taxiuri).

- Cazare:
Ședere la hotel pentru trei nopți la un hotel de afaceri de nivel mediu, aproape de locul întâlnirii.

- Mese:
Alocație zilnică pentru mese, pentru mic dejun, prânz și cină, bazată pe politica de diurnă a companiei.

- Cheltuieli diverse:
Taxe de parcare la aeroport.
Taxe pentru acces la internet la hotel.
Bacșișuri sau alte mici taxe de servicii.

- Documentație:
Trimiteți toate chitanțele (zboruri, taxiuri, hotel, mese etc.) și un raport complet de cheltuieli pentru rambursare.


## Importați bibliotecile necesare

Importați bibliotecile și modulele necesare pentru notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definește Modelele de Cheltuieli

 Creează un model Pydantic pentru cheltuielile individuale și o clasă ExpenseFormatter pentru a converti o interogare a utilizatorului în date structurate despre cheltuieli.

 Fiecare cheltuială va fi reprezentată în formatul:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generating the Email

Creați o funcție utilitară pentru a genera un e-mail pentru depunerea unei cereri de rambursare a cheltuielilor.
- Această unealtă folosește decoratorul `@tool` din Microsoft Agent Framework.
- Calculează suma totală a cheltuielilor și formatează detaliile într-un corp de e-mail.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Unealtă pentru extragerea cheltuielilor de călătorie din imagini cu chitanțe

Creează o funcție de unealtă pentru extragerea cheltuielilor de călătorie din imagini cu chitanțe.
- Această unealtă folosește decoratorul `@tool` din Microsoft Agent Framework.
- Aceasta citește imaginea chitanței, o encodează în base64 și returnează URI-ul datelor pentru ca agentul să analizeze.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Procesarea cheltuielilor

Definește agenții și conectează-i într-un flux de lucru secvențial folosind `WorkflowBuilder`.
- Agentul OCR extrage datele structurate despre cheltuieli din imaginea chitanței folosind instrumentul `load_receipt_image`.
- Agentul Email preia datele extrase și generează un email profesional pentru cererea de rambursare a cheltuielilor folosind instrumentul `generate_expense_email`.
- `WorkflowBuilder` cu `add_edge` creează un pipeline secvențial: Agent OCR → Agent Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Funcția principală

Construiește fluxul de lucru secvențial și rulează-l pentru a procesa imaginea chitanței și a genera e-mailul pentru cererea de decontare a cheltuielilor.


> **Notă:** Acest flux de lucru transmite în prezent imaginea chitanței ca text base64, pe care majoritatea modelelor de chat (inclusiv gpt-4o) nu îl vor trata ca o imagine.  
> Poate depăși, de asemenea, fereastra de context a modelului. Se recomandă rularea OCR cu Azure AI Vision (sau un alt instrument OCR) și trimiterea doar a textului extras, sau reconfigurarea pentru a trimite imaginea ca mesaj `image_url`.  
> Dacă doriți doar să evitați erorile de context, încercați o imagine a chitanței mai mică sau un model cu o fereastră de context mai mare.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
